In [ ]:
import pandas as pd
import random
from collections import defaultdict
import numpy as np

from sklearn.model_selection import train_test_split
random.seed(42)

🟢 1. LOAD DATA

In [ ]:
trans = pd.read_csv(
    OUTPUT_DIR / "transition_matrix_normalized_dedup.csv"
)

ppr = pd.read_csv(
    OUTPUT_DIR / "ppr_results_personalized_FAST_0.7.csv"
)
dpi_df = pd.read_csv(DATA_DIR / "evitdi_dpi_filtered_clean.csv")
ppi = pd.read_csv(DATA_DIR / "SIGNOR_ppi_GO_STRING.csv")

🟢 2. BUILD LOOKUPS

In [ ]:
p_uv = {(r.source, r.target): r.p_uv for r in trans.itertuples()}

ppr_dict = {}
for drug, sub in ppr.groupby("drugbank_id"):
    sub = sub[sub["node"] != drug]
    ppr_dict[drug] = dict(zip(sub["node"], sub["ppr_score"]))

ppi_lookup = {}
for r in ppi.itertuples():
    u, v = r.IDA, r.IDB
    conf, go = r.confidence_score, r.GO_norm
    ppi_lookup[(u, v)] = (conf, go)
    ppi_lookup[(v, u)] = (conf, go)

🔴 3. POSITIVE PAIRS

In [ ]:
pos_df = dpi_df[["drugbank_id", "uniprot_id"]].drop_duplicates()
pos_df["label"] = 1

🔴 4. BUILD POS TARGETS

In [ ]:
pos_targets = defaultdict(set)

for _, row in pos_df.iterrows():
    pos_targets[row["drugbank_id"]].add(row["uniprot_id"])

🔴 5. NEGATIVE SAMPLING (UNCHANGED)

In [ ]:
def sample_negative_target(drug, ppr_dict, pos_targets):
    if drug not in ppr_dict:
        return None
    candidates = [
        (n, s) for n, s in ppr_dict[drug].items()
        if n not in pos_targets.get(drug, set())
    ]
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda x: x[1])
    n = len(candidates)
    low  = candidates[:int(0.2*n)]
    mid  = candidates[int(0.2*n):int(0.8*n)]
    high = candidates[int(0.8*n):]

    r = random.random()
    if r < 0.2 and low:
        return random.choice([n for n, _ in low])
    elif r < 0.8 and mid:
        return random.choice([n for n, _ in mid])
    elif high:
        return random.choice([n for n, _ in high])

    return None

🔴 6. GENERATE NEGATIVE PAIRS

In [ ]:
neg_pairs = []
drugs = list(pos_targets.keys())

seen = set()

while len(neg_pairs) < len(pos_df):
    drug = random.choice(drugs)
    neg_target = sample_negative_target(drug, ppr_dict, pos_targets)

    if neg_target is None:
        continue

    pair = (drug, neg_target)
    if pair in seen:
        continue

    seen.add(pair)

    neg_pairs.append({
        "drugbank_id": drug,
        "uniprot_id": neg_target,
        "label": 0
    })

neg_df = pd.DataFrame(neg_pairs)

🔴 7. MERGE + SHUFFLE

In [ ]:
df = pd.concat([pos_df, neg_df], ignore_index=True)
df = df.drop_duplicates().sample(frac=1, random_state=42).reset_index(drop=True)

🔴 8. 🔥 RNDOM-WISE SPLIT (KEY CHANGE)

In [ ]:
# Random split (80/10/10)

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
save_dir = OUTPUT_DIR

train_df[["drugbank_id", "uniprot_id", "label"]].to_csv(
    f"{save_dir}/train_pairs_random.csv", index=False
)

val_df[["drugbank_id", "uniprot_id", "label"]].to_csv(
    f"{save_dir}/val_pairs_random.csv", index=False
)

test_df[["drugbank_id", "uniprot_id", "label"]].to_csv(
    f"{save_dir}/test_pairs_random.csv", index=False
)

🔴 9. BUILD GRAPH

In [ ]:
all_pos_pairs = df[df["label"] == 1]
neighbors = defaultdict(list)
# Drug-protein edges
for _, row in all_pos_pairs.iterrows():
    d = str(row["drugbank_id"])
    p = str(row["uniprot_id"])
    neighbors[d].append(p)
    neighbors[p].append(d)

# Filter proteins
filtered_trans = trans

# PPI edges
for _, r in filtered_trans.iterrows():
    src = str(r["source"])
    tgt = str(r["target"])
    neighbors[src].append(tgt)
    neighbors[tgt].append(src)
# Remove duplicates
for k in neighbors:
    neighbors[k] = list(set(neighbors[k]))

🔴 10. NEIGHBOR SELECTION (UNCHANGED)

In [ ]:
stats_deg = []
stats_kthr = []
stats_kfinal = []
stats_selected = []
def select_neighbors_entropy(v, nbrs, ppr_vec, tau, k_min, eps=1e-12):
    if not nbrs:
        return []
    # --- Step 1: Get raw PPR ---
    raw = {u: ppr_vec.get(u, 0.0) for u in nbrs}
    if all(score <= eps for score in raw.values()):
        return nbrs[:min(k_min, len(nbrs))]
    # --- Step 2: Normalize ---
    total = sum(raw.values()) + eps
    local = {u: raw[u] / total for u in raw}
    # --- Step 3: Sort by importance ---
    ordered = sorted(local.items(), key=lambda x: x[1], reverse=True)
    # --- Step 4: τ-based cumulative selection ---
    cum = 0.0
    k_thr = 0
    for _, w in ordered:
        cum += w
        k_thr += 1
        if cum >= tau:
            break
    # --- Step 5: Adaptive expansion (prevents collapse) ---
    deg = len(nbrs)
    # entropy = uncertainty of distribution
    import math
    entropy = -sum(p * math.log(p + eps) for p in local.values())
    entropy_norm = entropy / math.log(deg + eps)
    # --- Step 6: Final decision ---
    k_final = max( k_min, k_thr + max(1, int(entropy_norm * 2)))
    k_final = min(k_final, deg)
    stats_deg.append(deg)
    stats_kthr.append(k_thr)
    stats_kfinal.append(k_final)
    # --- Step 7: Select ---
    selected = [u for u, _ in ordered[:k_final]]
    stats_selected.append(len(selected))
    return selected

🔴 11. PATH EXTRACTION (UNCHANGED CORE)

In [ ]:
drug_nodes = {n for n in neighbors if str(n).startswith("DB")}


In [ ]:
def extract_paths_entropy(drug, target, ppr_vec, max_hops=5):

    drug, target = str(drug), str(target)
    paths = set()
    MAX_PATHS = 80

    def dfs(path):
        if len(paths) >= MAX_PATHS and random.random() < 0.7: return # Hard stop

        v = path[-1]
        # stop early if target is reached
        if v == target:
            if len(path) > 2:
                paths.add(tuple(path))
                return

        if len(path) - 1 >= max_hops: return

        # 1. Check for Target IMMEDIATELY
        nbrs = neighbors.get(v, [])

        if target in nbrs:
           if len(path) > 2:
              paths.add(tuple(path + [target]))

        # 2. Filtering & Selection
        valid_nbrs = [u for u in nbrs if u not in path and u not in drug_nodes]
        if not valid_nbrs: return

        # Use your entropy selection
        # Note: Reduced k_min to 3 to allow the "Branching Start" to handle diversity
        selected = select_neighbors_entropy(v, valid_nbrs, ppr_vec, tau=0.85, k_min=3)
        random.shuffle(selected)

        for u in selected:
            dfs(path + [u])
            if len(paths) >= MAX_PATHS: return

    # --- Step 8: THE MULTI-START (This fixes the Median) ---
    first_neighbors = neighbors.get(drug, [])
    random.shuffle(first_neighbors)

    # Force the search to start from at least 10 different drug neighbors
    for p in first_neighbors[:10]:
        if len(paths) >= MAX_PATHS and random.random( ) < 0.7: break
        dfs([drug, p])

    return paths

🔴 12. APPLY TO SPLITS




In [ ]:
def extract_paths_for_df(pairs_df):
    records = []
    for _, row in pairs_df.iterrows():
        drug = row["drugbank_id"]
        target = row["uniprot_id"]
        label = row["label"]
        ppr_vec = ppr_dict.get(drug, {})
        paths = extract_paths_entropy(drug, target, ppr_vec)
        if not paths:
            records.append({
                "drugbank_id": drug,
                "uniprot_id": target,
                "path": None,
                "path_length": 0,
                "label": label
            })
            continue

        for path in paths:
            records.append({
                "drugbank_id": drug,
                "uniprot_id": target,
                "path": " -> ".join(path),
                "path_length": len(path),
                "label": label
            })
    return pd.DataFrame(records)
train_paths_df = extract_paths_for_df(train_df)
val_paths_df   = extract_paths_for_df(val_df)
test_paths_df  = extract_paths_for_df(test_df)

🔴 14. SAVE

In [ ]:
train_paths_df.to_csv(
    OUTPUT_DIR / "train_paths_random.csv",
    index=False
)

val_paths_df.to_csv(
    OUTPUT_DIR / "val_paths_random.csv",
    index=False
)

test_paths_df.to_csv(
    OUTPUT_DIR / "test_paths_random.csv",
    index=False
)

In [ ]:
def build_all_tables(paths_df, split_name,
                     ppi_lookup, p_uv, ppr_lookup,
                     save_dir):

    import pandas as pd

    paths_df = paths_df.reset_index(drop=True)

    path_rows = []
    edge_rows = []
    node_rows = []

    for pid, r in paths_df.iterrows():

        drug = r["drugbank_id"]
        target = r["uniprot_id"]

        # =========================
        # CASE 1: NO PATH
        # =========================
        if pd.isna(r["path"]):

            path_rows.append({
                "path_id": pid,
                "drugbank_id": drug,
                "target_uniprot": target,
                "path_length": 0,
                "num_nodes": 0
            })

            node_rows.append({
                "path_id": pid,
                "node": None,
                "hop": -1,
                "ppr_score": 0.0,
                "node_role": "none"
            })

            continue

        # =========================
        # CASE 2: HAS PATH
        # =========================
        nodes = [n.strip() for n in r["path"].split("->") if n.strip()]

        # ---- PATH TABLE ----
        path_rows.append({
            "path_id": pid,
            "drugbank_id": drug,
            "target_uniprot": target,
            "path_length": r["path_length"],
            "num_nodes": len(nodes)
        })

        # ---- NODE TABLE ----
        for hop, node in enumerate(nodes):

            role = (
                "drug" if hop == 0 else
                "target" if hop == len(nodes) - 1 else
                "intermediate"
            )

            node_rows.append({
                "path_id": pid,
                "node": node,
                "hop": hop,
                "ppr_score": ppr_lookup.get((drug, node), 0.0),
                "node_role": role
            })

        # ---- EDGE TABLE ----
        for i in range(len(nodes) - 1):
            u, v = nodes[i], nodes[i + 1]

            conf, go = ppi_lookup.get(
                (u, v),
                ppi_lookup.get((v, u), (0.0, 0.0))
            )

            edge_rows.append({
                "path_id": pid,
                "hop": i + 1,
                "src": u,
                "dst": v,
                "transition_prob": p_uv.get(
                    (u, v),
                    p_uv.get((v, u), 0.0)
                ),
                "confidence_score": conf,
                "go_similarity": go
            })

    # =========================
    # SAVE
    # =========================
    path_df = pd.DataFrame(path_rows)
    edge_df = pd.DataFrame(edge_rows)
    node_df = pd.DataFrame(node_rows)

    path_df.to_csv(f"{save_dir}/{split_name}_path_table_random.csv", index=False)
    edge_df.to_csv(f"{save_dir}/{split_name}_edge_table_random.csv", index=False)
    node_df.to_csv(f"{save_dir}/{split_name}_node_table_random.csv", index=False)

    print(f"✅ {split_name} done:",
          path_df.shape, edge_df.shape, node_df.shape)

In [ ]:
save_dir = OUTPUT_DIR

build_all_tables(
    train_paths_df,
    "train",
    ppi_lookup,
    p_uv,
    ppr_dict,
    save_dir
)

build_all_tables(
    val_paths_df,
    "val",
    ppi_lookup,
    p_uv,
    ppr_dict,
    save_dir
)

build_all_tables(
    test_paths_df,
    "test",
    ppi_lookup,
    p_uv,
    ppr_dict,
    save_dir
)